# Mạng nơ-ron đồ thị · Từng bước
Đối chiếu ML Studio → Mạng nơ-ron đồ thị, buổi 1–4. Ví dụ trích dẫn là giả lập. Các cạnh được vô hướng hóa; trọng số GCN/GAT đặt sẵn. Mỗi hàng ma trận chỉ một nút đích và mỗi cột chỉ một nút nguồn.

**A.** Với A–B–C và đặc trưng 2, 4, 8, đoán giá trị A sau một và hai lượt mean có self-loop. Trước khi chạy, giải thích lúc nào C ảnh hưởng đến A.

Nguồn đọc: Liu & Zhou, §2.3, Ch. 4, §5.1.3, §7.1 và §11.1.

In [ ]:
import numpy as np

def adjacency(n, edges, self_loops=False):
    matrix = np.eye(n) if self_loops else np.zeros((n, n))
    for i, j in edges:
        if i == j or not (0 <= i < n and 0 <= j < n):
            raise ValueError("Cạnh phải nối hai nút hợp lệ.")
        matrix[i, j] = matrix[j, i] = 1
    return matrix

def mean_step(matrix, state):
    degree = matrix.sum(axis=1)
    received = matrix @ state
    return np.divide(received, degree, out=np.zeros_like(degree), where=degree > 0)

edges = [(0, 1), (1, 2)]
x = np.array([2., 4., 8.])
a = adjacency(len(x), edges, self_loops=True)
one = mean_step(a, x)
two = mean_step(a, one)
print(a.astype(int), one, two, sep="\n")
np.testing.assert_allclose(one, [3, 14/3, 6])
np.testing.assert_allclose(two[0], 23/6)

## Cập nhật đồng thời
Mọi nút đọc cùng trạng thái đầu lượt. Với chuỗi trên, C thiếu cạnh trực tiếp đến A, nên chỉ ảnh hưởng thông qua B ở lượt hai.

**B.** Tắt self-loop và thêm cạnh A–C. Tính vector của lượt đầu bằng tay rồi điền `answer`. Chạy lại ô để đối chiếu.

In [ ]:
triangle = adjacency(3, edges + [(0, 2)])
answer = None
if answer is None:
    print("CHƯA HOÀN THÀNH · Điền vector dự đoán.")
else:
    np.testing.assert_allclose(answer, mean_step(triangle, x))
    print("ĐẠT · Vector của em khớp phép tính.")

## GCN và GAT
GCN đối xứng chuẩn hóa mỗi cạnh theo bậc của cả nút nhận và nút gửi. Trong ví dụ GAT, hệ số được tính bằng softmax các điểm LeakyReLU trên từng hàng. Cạnh không tồn tại có hệ số 0. Trên web, nút cô lập khi tắt self-loop trả tổng bằng 0.

In [ ]:
def gcn_weights(matrix):
    degree = matrix.sum(axis=1)
    inv = np.divide(1., np.sqrt(degree), out=np.zeros_like(degree), where=degree > 0)
    return inv[:, None] * matrix * inv[None, :]

def gat_weights(matrix, state, w=.8):
    projected = w * state
    raw = .6 * projected[:, None] - .3 * projected[None, :]
    scores = np.where(raw < 0, .2 * raw, raw)
    weights = np.zeros_like(matrix, dtype=float)
    for i, row in enumerate(matrix):
        active = row.astype(bool)
        if active.any():
            exps = np.exp(scores[i, active] - scores[i, active].max())
            weights[i, active] = exps / exps.sum()
    return weights

lab_x = np.array([2., 4., 8., 6., 1.])
lab_a = adjacency(5, [(0,1), (1,2), (2,3), (3,4)], self_loops=True)
gcn = gcn_weights(lab_a)
gat = gat_weights(lab_a, lab_x)
print("GCN nút A:", (gcn @ (.8 * lab_x))[0])
print("GAT nút A:", (gat @ (.8 * lab_x))[0])
np.testing.assert_allclose(gcn[0, :2], [.5, 1 / np.sqrt(6)])
np.testing.assert_allclose(gat.sum(axis=1), 1)
np.testing.assert_array_equal(gat[lab_a == 0], 0)

**C.** Thêm A–C trên web và trong NumPy. Ghi lại các hệ số cũ và mới của A. Đổi thứ tự nút trong cả ma trận và vector để kiểm chứng rằng kết quả chỉ đổi thứ tự tương ứng.

Gợi ý bài B: mỗi nút trong tam giác nhìn hai nút còn lại. Đáp án sau khi thử: [6, 5, 3].